# Sub-project 3: Trích xuất thông tin món ăn bằng Qwen3-VL

Model chạy trực tiếp trong Colab qua `transformers`. Không cần server hay API key. Mỗi ảnh cho ra một JSON gồm `dish_name`, `cuisine`, `main_ingredients`, `cooking_method`, `confidence`.

**GPU:** L4 hoặc A100 chạy bf16. T4 tự chuyển sang 4-bit (`bitsandbytes`). Bảng so sánh với classifier dùng các run `image`, `text_strict`, `xattn_strict` của Mốc 2 (`vlm.compare_runs`); run nào chưa có sẽ bị bỏ qua.

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/viethoang2503/food-recognize-extract.git"
REPO_DIR = Path("/content/food-recognize-extract")
WORK_DIR = Path("/content/drive/MyDrive/foodmm")
DATA_ROOT = Path("/content/data/upmc_food101")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
WORK_DIR.mkdir(parents=True, exist_ok=True)
BASE = [f"paths.data_root={DATA_ROOT}", f"paths.work_dir={WORK_DIR}"]
print("repo:", REPO_DIR, "| work dir:", WORK_DIR)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
def run(script, *args):
    """Run a repo script and stream its output into the notebook."""
    cmd = [sys.executable, "-u", f"scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{script} failed with exit code {proc.returncode}")

## Dữ liệu
Giải nén bản zip trên Drive về `/content` (lần đầu tải từ Kaggle, cần Colab Secret `KAGGLE_API_TOKEN`, hoặc cặp `KAGGLE_USERNAME` + `KAGGLE_KEY`). `prepare_data.py` bỏ qua nếu manifest đã có từ Mốc 1.

In [ ]:
import zipfile
from google.colab import userdata

ZIP_PATH = WORK_DIR / "data" / "upmcfood101.zip"
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    from google.colab.userdata import SecretNotFoundError
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)  # new token style needs a recent client
    try:  # new-style token (KGAT_...) first, then the legacy username + key pair
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except SecretNotFoundError:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    subprocess.run(["kaggle", "datasets", "download", "-d", "gianmarco96/upmcfood101", "-p", str(ZIP_PATH.parent)], check=True)
if not DATA_ROOT.exists():
    DATA_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
run("prepare_data.py", "--set", *BASE)

## 1. Chạy thử 5 ảnh
Nạp model (lần đầu tải khoảng 9 GB về `/root/.cache`) và kiểm tra output. Kết quả ghi vào `foodmm/vlm/smoke`, tách khỏi lần chạy thật.

In [ ]:
import torch

print(torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
SMOKE = ["--set", *BASE, "vlm.tag=smoke", "vlm.n_samples=5"]
run("vlm_extract.py", "--mode", "image", *SMOKE)
print((WORK_DIR / "vlm" / "smoke" / "extract_image.jsonl").read_text()[:1500])

## 2. Chạy 200 ảnh với 3 chế độ prompt
`image` (chỉ ảnh), `text` (chỉ text đã che `strict`, không gửi ảnh), `image_text` (ảnh + text đã che): bộ so sánh image-only / text-only / multimodal của đề bài. Mất session thì chạy lại cell, các ảnh đã xong được bỏ qua. Có thể thêm `vlm.model_name=Qwen/Qwen3-VL-8B-Instruct vlm.tag=qwen3vl8b` để thử bản 8B.

In [ ]:
for mode in ("image", "text", "image_text"):
    run("vlm_extract.py", "--mode", mode, "--set", *BASE)

## 3. Đánh giá
`dish_acc_lo`/`dish_acc_hi` là khoảng tin cậy 95% (bootstrap). Bảng thứ hai là accuracy của các classifier trên đúng 200 ảnh này, để so sánh trực tiếp. `ingredient_grounding` chỉ để tham khảo: các chế độ có `uses_text=True` được đọc chính text dùng để đo nên điểm bị thổi phồng.

In [ ]:
from IPython.display import Markdown, display

run("vlm_evaluate.py", "--set", *BASE)
VLM_DIR = WORK_DIR / "vlm" / "qwen3vl4b"
display(Markdown((VLM_DIR / "eval_summary.md").read_text()))
display(Markdown("### Classifier trên cùng mẫu\n" + (VLM_DIR / "classifier_on_sample.md").read_text()))

## 4. Xem vài ví dụ

In [ ]:
import json

import matplotlib.pyplot as plt
from PIL import Image as PILImage

from foodmm.data.prepare import load_manifest
from foodmm.vlm.evaluate import map_dish_to_class
from foodmm.vlm.extract import read_records
from foodmm.utils import load_json

manifest = load_manifest(WORK_DIR / "data" / "manifest.csv").set_index("id")
classes = load_json(WORK_DIR / "data" / "classes.json")
recs = {m: {r["id"]: r for r in read_records(VLM_DIR / f"extract_{m}.jsonl")}
        for m in ("image", "text", "image_text")}
for sid in load_json(VLM_DIR / "sample_ids.json")[:6]:
    row = manifest.loc[sid]
    plt.figure(figsize=(3, 3))
    plt.imshow(PILImage.open(DATA_ROOT / row["image_path"]).convert("RGB"))
    plt.axis("off")
    plt.title(row["label"])
    plt.show()
    for mode, by_id in recs.items():
        r = by_id.get(sid)
        if r and r["valid"]:
            print(f"[{mode}] -> {map_dish_to_class(r['output']['dish_name'], classes)}")
            print(json.dumps(r["output"], ensure_ascii=False))
        elif r:
            print(f"[{mode}] invalid: {r['error']}")

## 5. Chấm tay
1. Mở `MyDrive/foodmm/vlm/qwen3vl4b/manual_grading.csv` (Google Sheets hoặc Excel). Các dòng đã được xáo trộn và **không ghi chế độ prompt**, để chấm mù. Đừng mở `manual_grading_key.csv` trước khi chấm xong.
2. Điền `grade_dish` (0/1), `grade_ingredients` (0: sai, 1: một phần, 2: tốt), `grade_method` (0/1). Có thể ghi chú ở `notes`.
3. Lưu lại dưới dạng CSV cùng tên, rồi chạy cell dưới để tính điểm trung bình theo chế độ.

In [ ]:
run("vlm_evaluate.py", "--set", *BASE)
scores = VLM_DIR / "manual_scores.csv"
print(scores.read_text() if scores.exists() else "Chưa có dòng nào được chấm.")